In [1]:
# ════════════════════════════════════════════════════════════════════
# CELL 1: IMPORT
# ════════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/home/duoc/workflow/api')

import requests
from io import BytesIO
from PIL import Image

from app.rag.service import Embedder
from app.rag.image_rag import ImageRAG

embedder = Embedder()
print('✅ Import OK')

INFO:rag:[config] device=cuda dtype=torch.float16
INFO:rag:[tokenizer] fallback regex


✅ Import OK


In [2]:
# ════════════════════════════════════════════════════════════════════
# CELL 2: ADD ẢNH TỪ URL
# ════════════════════════════════════════════════════════════════════
print('=' * 60)
print('CELL 2: ADD ẢNH TỪ URL')
print('=' * 60)

IMAGE_URL = 'https://scontent.fdad3-6.fna.fbcdn.net/v/t39.30808-6/486603446_1084903300320859_6011244534783207630_n.jpg?stp=c0.119.1080.1080a_cp6_dst-jpg_tt6&cstp=mx1080x1080&ctp=s206x206&_nc_cat=102&ccb=1-7&_nc_sid=50ad20&_nc_ohc=3p4EmFBFTGkQ7kNvwELroRz&_nc_oc=AdpOdKniczqSKFaOCfzNF68ElA_oU78hbVkDqOgCww0Y4PAvKXYr95FERdJXJJVmZco&_nc_zt=23&_nc_ht=scontent.fdad3-6.fna&_nc_gid=0A3abGgfwgrm3rgioN4q9A&_nc_ss=7b289&oh=00_Af88ck7qBcG4LSI6byGjnQpCbMb--joO2z2_Rpovpsdfmw&oe=6A3FE8DD'

rag = ImageRAG(embedder=embedder)

async def add_image():
    print(f'📥 Tải ảnh từ: {IMAGE_URL[:60]}...')
    resp = requests.get(IMAGE_URL, timeout=30)
    resp.raise_for_status()
    img = Image.open(BytesIO(resp.content))
    print(f'   Original size: {img.size}')

    img.thumbnail((448, 448))
    if img.mode != 'RGB':
        img = img.convert('RGB')
    print(f'   Resized: {img.size}')

    # Test caption riêng
    print('\n📝 Florence caption:')
    caption = await rag._captioner.caption(img)
    print(f'   {caption}')

    result = await rag.add(
        image=img,
        image_id='test_001',
        source_url=IMAGE_URL,
        tags=['test'],
    )
    print(f'\n   Result : {result}')
    print(f'   Stats  : {rag.stats()}')
    if rag._store and rag._store._metas:
        print(f'   Meta   : {rag._store._metas[-1]}')

await add_image()

INFO:rag:[caption] loading /home/duoc/models/florence-2-large...


CELL 2: ADD ẢNH TỪ URL
📥 Tải ảnh từ: https://scontent.fdad3-6.fna.fbcdn.net/v/t39.30808-6/4866034...
   Original size: (206, 206)
   Resized: (206, 206)

📝 Florence caption:


`torch_dtype` is deprecated! Use `dtype` instead!
INFO:rag:[caption] loaded
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
INFO:rag:[caption] unloaded, VRAM freed
INFO:rag:[image_load] 5 images restored
INFO:rag:[caption] loading /home/duoc/models/florence-2-large...


   A plate of food with a variety of ingredients. On the plate, there are several small, round, brown and white seashells arranged in a circular pattern. The shells appear to be cooked and have a glossy texture. They are surrounded by a bed of green leafy vegetables, possibly lettuce or arugula. The plate is white and has a textured surface. The background is blurred, but it appears to be a table or countertop.


INFO:rag:[caption] loaded
INFO:rag:[caption] unloaded, VRAM freed



   Result : duplicate
   Stats  : {'images': 5, 'faiss_vectors': 5}
   Meta   : {'image_id': 'trash_001', 'caption': 'The image is a plain white background. It appears to be a simple, minimalistic image with no other elements or decorations. The image is centered in the middle of the image and has a smooth, clean surface. The background is a solid white color and there are no shadows or reflections on it.'}


In [3]:
# ════════════════════════════════════════════════════════════════════
# CELL 3: ADD NHIỀU ẢNH ĐỂ TEST LỌC RÁC
# ════════════════════════════════════════════════════════════════════
print('=' * 60)
print('CELL 3: ADD NHIỀU ẢNH')
print('=' * 60)

TEST_IMAGES = [
    ('food_001',   'https://scontent.fdad3-6.fna.fbcdn.net/v/t39.30808-6/486961473_1084916220319567_2979543827476588806_n.jpg?stp=c0.119.1080.1080a_dst-jpg_tt6&cstp=mx1080x1080&ctp=s206x206&_nc_cat=105&ccb=1-7&_nc_sid=50ad20&_nc_ohc=OCeImb6wGnAQ7kNvwHONm6P&_nc_oc=AdowSorZ9px8hkrjsVx-5MwF1uIBa_KbPFKkVE6g1lO1az19gviixMoRmlxAhqV_iGk&_nc_zt=23&_nc_ht=scontent.fdad3-6.fna&_nc_gid=-_KfInJbzhMazdL44LUKfQ&_nc_ss=7b289&oh=00_Af-BKxqYhz32cbQixkhgFU07JU3tQ0RFgLflG6ROu_VsBA&oe=6A3FDD3C'),
    ('room_001',   'https://scontent.fdad3-6.fna.fbcdn.net/v/t39.30808-6/486850011_1084913230319866_290924985191983023_n.jpg?stp=c160.0.640.640a_dst-jpg_tt6&cstp=mx640x640&ctp=s206x206&_nc_cat=110&ccb=1-7&_nc_sid=50ad20&_nc_ohc=WRR5p08elpsQ7kNvwGEE-AH&_nc_oc=Adrmuegc8Jdd7-tKSHhFSrGPFwe37vflzHmT5hkTgsDdCYzQ2AzVZKAdI9GRF_2AEZ8&_nc_zt=23&_nc_ht=scontent.fdad3-6.fna&_nc_gid=7u1t006AlhkyGB-csT3cuA&_nc_ss=7b289&oh=00_Af81iqg9QB_ck1p9baoZZGE8IbFOwYqNu5_Cn7yGL9KstA&oe=6A3FDD89'),
    ('nature_001', 'https://scontent.fdad3-6.fna.fbcdn.net/v/t39.30808-6/486250306_1084903063654216_2736098380315280398_n.jpg?stp=c179.0.1082.1082a_cp6_dst-jpg_tt6&cstp=mx1082x1082&ctp=s206x206&_nc_cat=108&ccb=1-7&_nc_sid=50ad20&_nc_ohc=g9jbJcVM4VQQ7kNvwE-dk3j&_nc_oc=AdopMNOrkCpVgpktEIeafzThXvP6JCBdTYHq8DKTVHy1zo4BpeQ7rbBX3O5Pm0Heb-Y&_nc_zt=23&_nc_ht=scontent.fdad3-6.fna&_nc_gid=0A3abGgfwgrm3rgioN4q9A&_nc_ss=7b289&oh=00_Af_dK4Y-DNzez6FvnBotto-euoTvKmBdlnHf-ll82mnxxQ&oe=6A3FF28F'),
]

async def add_batch():
    for image_id, url in TEST_IMAGES:
        print(f'\n📥 {image_id}')
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            img = Image.open(BytesIO(resp.content))
            img.thumbnail((448, 448))
            if img.mode != 'RGB':
                img = img.convert('RGB')
            result = await rag.add(image=img, image_id=image_id)
            print(f'   → {result}')
        except Exception as e:
            print(f'   ❌ Lỗi: {e}')

    print(f'\n📊 Stats: {rag.stats()}')

await add_batch()

CELL 3: ADD NHIỀU ẢNH

📥 food_001
   → duplicate

📥 room_001
   → duplicate

📥 nature_001
   → duplicate

📊 Stats: {'images': 5, 'faiss_vectors': 5}


In [8]:
# ════════════════════════════════════════════════════════════════════
# CELL 4: SEARCH BẰNG TEXT
# ════════════════════════════════════════════════════════════════════
print('=' * 60)
print('CELL 4: SEARCH BẰNG TEXT')
print('=' * 60)

QUERIES = [
    'pasta food dinner',
    'hotel room bed',
    'nature landscape',
    'seafood crab',
    'street food',
]

async def test_search():
    for q in QUERIES:
        print(f'\n🔍 "{q}"')
        results = await rag.search_by_text(q, k=3)
        if not results:
            print('   ❌ Không có kết quả')
            continue
        for r in results:
            score   = r['score']
            meta    = r.get('meta', {})
            caption = meta.get('caption', 'N/A')[:80]
            bar     = '█' * int(score * 10) + '░' * (10 - int(score * 10))
            print(f'   {bar} {score:.3f} | {r["image_id"]} | {caption}')

await test_search()

CELL 4: SEARCH BẰNG TEXT

🔍 "pasta food dinner"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   █████░░░░░ 0.563 | nature_001 | The image shows a black frying pan with a dish in it. The dish appears to be a t
   █████░░░░░ 0.553 | test_001 | The image shows a plate of food with a variety of ingredients. On the plate, the
   █████░░░░░ 0.548 | food_001 | The image shows a table with a variety of food items on it. On the left side of 

🔍 "hotel room bed"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   ███░░░░░░░ 0.389 | room_001 | The image shows two young women standing in front of a table with a large cake o
   ███░░░░░░░ 0.384 | test_001 | The image shows a plate of food with a variety of ingredients. On the plate, the
   ███░░░░░░░ 0.379 | food_001 | The image shows a table with a variety of food items on it. On the left side of 

🔍 "nature landscape"


...

   ████░░░░░░ 0.482 | test_001 | The image shows a plate of food with a variety of ingredients. On the plate, the
   ████░░░░░░ 0.437 | food_001 | The image shows a table with a variety of food items on it. On the left side of 
   ████░░░░░░ 0.419 | room_001 | The image shows two young women standing in front of a table with a large cake o

🔍 "seafood crab"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   █████░░░░░ 0.505 | test_001 | The image shows a plate of food with a variety of ingredients. On the plate, the
   ████░░░░░░ 0.407 | food_001 | The image shows a table with a variety of food items on it. On the left side of 
   ███░░░░░░░ 0.398 | nature_001 | The image shows a black frying pan with a dish in it. The dish appears to be a t

🔍 "street food"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   █████░░░░░ 0.550 | test_001 | The image shows a plate of food with a variety of ingredients. On the plate, the
   █████░░░░░ 0.544 | food_001 | The image shows a table with a variety of food items on it. On the left side of 
   ████░░░░░░ 0.499 | nature_001 | The image shows a black frying pan with a dish in it. The dish appears to be a t


In [9]:
# ════════════════════════════════════════════════════════════════════
# CELL 5: TEST DUPLICATE VÀ TRASH
# ════════════════════════════════════════════════════════════════════
print('=' * 60)
print('CELL 5: TEST DUPLICATE & TRASH')
print('=' * 60)

from PIL import Image as PILImage
import numpy as np

async def test_edge_cases():
    # Test duplicate
    print('\n[1] Duplicate (cùng image_id):')
    resp = requests.get(TEST_IMAGES[0][1], timeout=30)
    img = Image.open(BytesIO(resp.content)).convert('RGB')
    img.thumbnail((448, 448))
    result = await rag.add(image=img, image_id='food_001')
    print(f'   → {result}  (expected: duplicate)')

    # Test ảnh rác (ảnh trắng)
    print('\n[2] Ảnh rác (solid white):')
    blank = PILImage.fromarray(np.ones((100, 100, 3), dtype=np.uint8) * 255)
    result = await rag.add(image=blank, image_id='trash_001')
    print(f'   → {result}  (expected: rejected)')

    print(f'\n📊 Stats sau test: {rag.stats()}')

await test_edge_cases()

CELL 5: TEST DUPLICATE & TRASH

[1] Duplicate (cùng image_id):
   → duplicate  (expected: duplicate)

[2] Ảnh rác (solid white):


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:rag:[image_add] trash_001 | caption: The image is a plain white background. It appears to be a si...


   → ok  (expected: rejected)

📊 Stats sau test: {'images': 5, 'faiss_vectors': 5}


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

True
NVIDIA GeForce GTX 970
3.93560791015625 GB


In [2]:
import torch
import time
import requests
from io import BytesIO
from PIL import Image
import numpy as np

IMAGE_URL = 'https://scontent.fdad3-6.fna.fbcdn.net/v/t39.30808-6/486603446_1084903300320859_6011244534783207630_n.jpg?stp=c0.119.1080.1080a_cp6_dst-jpg_tt6&cstp=mx1080x1080&ctp=s206x206&_nc_cat=102&ccb=1-7&_nc_sid=50ad20&_nc_ohc=3p4EmFBFTGkQ7kNvwELroRz&_nc_oc=AdpOdKniczqSKFaOCfzNF68ElA_oU78hbVkDqOgCww0Y4PAvKXYr95FERdJXJJVmZco&_nc_zt=23&_nc_ht=scontent.fdad3-6.fna&_nc_gid=0A3abGgfwgrm3rgioN4q9A&_nc_ss=7b289&oh=00_Af88ck7qBcG4LSI6byGjnQpCbMb--joO2z2_Rpovpsdfmw&oe=6A3FE8DD'

# Load ảnh
resp = requests.get(IMAGE_URL, timeout=30)
img = Image.open(BytesIO(resp.content))
img.thumbnail((448, 448))
if img.mode != 'RGB':
    img = img.convert('RGB')
print(f"Ảnh size: {img.size}")

from app.rag.image_rag import ImageCaptioner
from app.rag.service import Embedder

captioner = ImageCaptioner()
embedder = Embedder()

print("\n" + "="*50)
print("FLORENCE-2 CAPTION")
print("="*50)
vram_before = torch.cuda.memory_allocated()/1024**3
t0 = time.perf_counter()
caption = await captioner.caption(img)
t1 = time.perf_counter()
vram_after = torch.cuda.memory_allocated()/1024**3

print(f"Caption : {caption}")
print(f"Time    : {t1-t0:.2f}s")
print(f"VRAM    : {vram_before:.2f} GB → {vram_after:.2f} GB (+{vram_after-vram_before:.2f} GB)")

print("\n" + "="*50)
print("BGE-M3 EMBED")
print("="*50)
vram_before2 = torch.cuda.memory_allocated()/1024**3
t2 = time.perf_counter()
emb = await embedder.encode([caption])
t3 = time.perf_counter()
vram_after2 = torch.cuda.memory_allocated()/1024**3

print(f"Vector shape : {np.array(emb).shape}")
print(f"Time         : {t3-t2:.2f}s")
print(f"VRAM         : {vram_before2:.2f} GB → {vram_after2:.2f} GB (+{vram_after2-vram_before2:.2f} GB)")

print("\n" + "="*50)
print("TỔNG")
print("="*50)
print(f"Caption + Embed : {t1-t0+t3-t2:.2f}s")
print(f"VRAM tổng dùng  : {vram_after2:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

INFO:rag:[caption] loading /home/duoc/models/florence-2-base...


Ảnh size: (206, 206)

FLORENCE-2 CAPTION


`torch_dtype` is deprecated! Use `dtype` instead!
INFO:rag:[caption] loaded
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
INFO:rag:[caption] unloaded, VRAM freed
INFO:rag:[embed] loading BAAI/bge-m3...
INFO:sentence_transformers.base.model:No device provided, using cuda:0
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from BAAI/bge-m3.


Caption : A plate of clams on a white plate. The clams are brown and white in color and have a pattern of small spots on their shells. They are arranged in a pile on top of a bed of white rice. There are also some green leafy vegetables scattered around the plate, including a few star-shaped leaves. The plate is sitting on a beige tablecloth.
Time    : 4.15s
VRAM    : 0.00 GB → 0.01 GB (+0.01 GB)

BGE-M3 EMBED


INFO:rag:[embed] loaded


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Vector shape : (1, 1024)
Time         : 1.90s
VRAM         : 0.01 GB → 2.12 GB (+2.12 GB)

TỔNG
Caption + Embed : 6.05s
VRAM tổng dùng  : 2.12 GB / 3.9 GB


In [2]:
import torch
import time
import requests
from io import BytesIO
from pathlib import Path
from PIL import Image

from app.rag.image_rag import ImageCaptioner

# ============================================================
# NHẬP THỦ CÔNG 10 ẢNH TẠI ĐÂY
# Có thể là path local (ví dụ: "/home/claude/test1.jpg")
# hoặc URL (http://...)
# ============================================================
IMAGE_SOURCES = [
    "./img/anh1.jpg",
    "./img/anh2.jpg",
    "./img/anh3.jpg",
    "./img/anh4.jpg",
    "./img/anh5.jpg",
    "./img/anh6.jpg",
    "./img/anh7.jpg",
    "./img/anh8.jpg",
    "./img/anh9.jpg",
    "./img/anh10.jpg",
]

def load_image(src: str) -> Image.Image:
    if src.startswith(("http://", "https://")):
        resp = requests.get(src, timeout=30)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content))
    else:
        img = Image.open(Path(src))
    img.thumbnail((448, 448))
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img


async def main():
    images = [load_image(src) for src in IMAGE_SOURCES]

    # ===== STAGE 1: CAPTION (Florence-2 load 1 lần cho cả lô) =====
    print("=" * 60)
    print(f"STAGE 1: FLORENCE-2 CAPTION — {len(images)} ẢNH")
    print("=" * 60)

    captioner = ImageCaptioner()
    t0 = time.perf_counter()
    captions = await captioner.caption_batch(images)   # <-- dùng caption_batch, KHÔNG dùng caption
    t1 = time.perf_counter()

    for src, cap in zip(IMAGE_SOURCES, captions):
        print(f"\n{src}\n  -> {cap}")

    print(f"\n>> Stage 1: {t1-t0:.2f}s tổng | {(t1-t0)/len(images):.2f}s/ảnh trung bình")
    print(f">> VRAM sau unload: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

    print("=" * 60)



await main()
    

INFO:rag:[caption] loading /home/duoc/models/florence-2-base...


STAGE 1: FLORENCE-2 CAPTION — 10 ẢNH


`torch_dtype` is deprecated! Use `dtype` instead!
INFO:rag:[caption] loaded
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
INFO:rag:[caption] unloaded, VRAM freed



./img/anh1.jpg
  -> A group of nine people sitting around a long wooden table in a restaurant. They are all smiling and appear to be enjoying a meal together. The table is set with plates, bowls, glasses, and cutlery. In the background, there is a large window that offers a view of the city skyline at night. The restaurant has a modern design with a yellow and orange color scheme.

./img/anh2.jpg
  -> A wooden serving board with a large platter of food on it. The platter is filled with a variety of ingredients, including shrimp, noodles, and a salad. The shrimp are cooked and appear to be seasoned with herbs and spices. The noodles are mixed with a mix of greens, red peppers, and other vegetables. There are also small bowls of sauce and condiments on the table. In the background, there are two glasses of beer and a few people sitting at a table. The overall atmosphere of the image is casual and relaxed.

./img/anh3.jpg
  -> The exterior of a restaurant named "Cálua" at night. The rest